# Part 1

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
data = pd.read_csv("mushroom/agaricus-lepiota.data", header=None)

y = data.iloc[:, 0]
X = data.iloc[:, 1:]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [5]:
P_edible = np.mean(y_train == 'e')
P_poisonous = np.mean(y_train == 'p')

print("P(Y = Edible):", P_edible)
print("P(Y = Poisonous):", P_poisonous)

P(Y = Edible): 0.5199409158050221
P(Y = Poisonous): 0.48005908419497784


In [7]:
cond_prob = {}

classes = ['e', 'p']

for c in classes:
    cond_prob[c] = {}
    
    X_c = X_train[y_train == c]
    n_c = len(X_c)
    
    for col in X_train.columns:
        cond_prob[c][col] = {}
        
        values = X_train[col].unique()
        k = len(values)
        
        for v in values:
            count = np.sum(X_c[col] == v)
            
            prob = (count + 1) / (n_c + k)
            
            cond_prob[c][col][v] = prob

# Part 2

In [11]:
def predict_nb(X, cond_prob, P_edible, P_poisonous):
    predictions = []
    
    for _, row in X.iterrows():
        log_prob_e = np.log(P_edible)
        log_prob_p = np.log(P_poisonous)
        
        for col in X.columns:
            val = row[col]
            
            log_prob_e += np.log(cond_prob['e'][col][val])
            log_prob_p += np.log(cond_prob['p'][col][val])
        
        if log_prob_e > log_prob_p:
            predictions.append('e')
        else:
            predictions.append('p')
    
    return np.array(predictions)

In [10]:
y_pred = predict_nb(X_test, cond_prob, P_edible, P_poisonous)

print(y_pred[:10])

['e' 'p' 'p' 'e' 'p' 'p' 'p' 'p' 'e' 'e']


# Part 3

In [12]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [13]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label='p')
recall = recall_score(y_test, y_pred, pos_label='p')
f1 = f1_score(y_test, y_pred, pos_label='p')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

Accuracy: 0.9488
Precision: 0.9901
Recall: 0.9041
F1 Score: 0.9451
